# 1.2 - Data cleaning (`accidentes.csv`)

Goal: produce a clean `accidentes` table and a short data report for the 8 Oct meeting.

Scope (M1 only):
1. Normalise categoricals (case / whitespace / accents) and merge synonyms.
2. Map junk values `(En blanco)`, `0`, `_`, empty → NaN.
3. Numeric fixes: `-1` → NaN in count columns; parse `fecha` / `hora`.
4. Add label `severe` (keep injury columns in the file — **leakage**, never use as features).

In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/accidentes.csv")
OUT_PATH = Path("./accidentes_clean.csv")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print("loaded:", DATA_PATH.resolve())
print("shape:", df.shape)
df.head()

loaded: /Users/lukas/dev/UC3M/Project-UC3M/data/accidentes.csv
shape: (2062, 21)


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,motos_implicadas,pesados_implicados,estado_firme,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id
0,01/01/2024 0:00,1:30:48,20240101-0001,08NL10,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,NaN,1,0,0,SEÑALIZAR ACCIDENTE POR ALCANCE EN CARRIL CENT...,CENTRO,49.0
1,01/01/2024 0:00,1:59:17,20240101-0002,07NC10,SUPERFICIE,M30,NC,Via de Servicio,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,seco,FLUIDA,0,0,0,"AVISO DE ALCANCE ENTRE 2 VEHÍCULOS, ANTES DE L...",USERA,NaN
2,01/01/2024 0:00,5:58:11,20240101-0007,03FT45,TÚNEL,A5,FT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,FLUIDA,0,0,0,ACCIDENTE LEVE ENTRE DOS VEHICULOS (ALCANCE) E...,VICALVARO,29.0
3,01/02/2024,9:51:53,20240102-0009,30RJ10,SUPERFICIE,M30,30RJ,RAMAL,Salida de via,COLISIÓN MEDIANA,...,0,0,Seco y limpio,FLUIDA,0,0,0,"ACCIDENTE LEVE SIN HERIDOS,EL VEHÍCULO CHOCA C...",RETIRO,NaN
4,02/01/2024 0:00,18:14:56,20240102-0022,11NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,DENSA,0,0,0,SEÑALIZAR COLISIÓN POR ALCANCE ENTRE DOS VEHÍC...,ARGANZUELA,50.0


## Helpers

Shared normalisation + junk handling used by every categorical clean.

In [2]:
JUNK = {"(en blanco)", "(enblanco)", "0", "_", "", "nan", "none", "null", "na", "n/a"}


def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_key(x) -> str | None:
    """Lowercase, strip accents/whitespace; return None for junk/missing."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    key = re.sub(r"\s+", " ", strip_accents(s).lower())
    key = key.replace("(en blanco)", "(en blanco)")
    if key in JUNK:
        return None
    return key


def apply_map(series: pd.Series, mapping: dict) -> pd.Series:
    """Map via normalised key → canonical label; unmapped non-junk kept as stripped original."""
    out = []
    for v in series:
        k = norm_key(v)
        if k is None:
            out.append(np.nan)
        elif k in mapping:
            out.append(mapping[k])
        else:
            out.append(str(v).strip())
    return pd.Series(out, index=series.index, dtype="object")


def before_after(df_before, df_after, cols):
    rows = []
    for c in cols:
        rows.append({
            "column": c,
            "nunique_before": int(df_before[c].nunique(dropna=False)),
            "nunique_after": int(df_after[c].nunique(dropna=False)),
            "missing_before": int(df_before[c].isna().sum()),
            "missing_after": int(df_after[c].isna().sum()),
        })
    return pd.DataFrame(rows)


CLEAN_COLS = [
    "estado_firme", "tipo_via", "tipo_colision", "tipo_accidente",
    "ubicacion", "circulacion", "carretera", "distrito",
]
snapshot_before = {c: df[c].copy() for c in CLEAN_COLS if c in df.columns}
print("helpers ready; columns to clean:", list(snapshot_before))

helpers ready; columns to clean: ['estado_firme', 'tipo_via', 'tipo_colision', 'tipo_accidente', 'ubicacion', 'circulacion', 'carretera', 'distrito']


## 1. Inspect unique values (before cleaning)

Confirm dirty variants before applying explicit maps.

In [3]:
for c in CLEAN_COLS:
    if c not in df.columns:
        continue
    vc = df[c].astype("object").value_counts(dropna=False)
    print(f"\n=== {c} ({vc.shape[0]} levels) ===")
    for val, n in vc.items():
        print(f"  {n:4d} | {val!r}")


=== estado_firme (13 levels) ===
  1618 | 'Seco y limpio'
   196 | 'Mojado'
    50 | nan
    40 | 'Seco'
    37 | 'SECO'
    32 | 'seco'
    32 | 'Otros'
    30 | 'SECO Y LIMPIO'
     9 | 'Húmedo'
     8 | 'MOJADO'
     6 | 'mojado'
     2 | '(En blanco)'
     2 | 'HUMEDO'

=== tipo_via (12 levels) ===
   825 | 'TRONCO'
   685 | 'VIA DE SERVICIO'
   416 | 'RAMAL'
    20 | 'Ramal'
    19 | 'TRANSFER'
    17 | 'TABLERO'
    16 | 'Via de Servicio'
    15 | 'Transfer'
    14 | 'tronco'
    12 | 'Túnel'
    12 | 'TUNEL'
    11 | 'Tronco'

=== tipo_colision (6 levels) ===
  1149 | 'COLISIÓN POR ALCANCE'
   393 | 'Otro'
   315 | 'COLISIÓN LATERAL'
    92 | 'COLISIÓN MEDIANA'
    80 | nan
    33 | 'COLISIÓN FRONTAL'

=== tipo_accidente (6 levels) ===
  1498 | 'Colision entre vehiculos'
   361 | 'Salida de via'
   173 | 'Otros'
    14 | 'Vuelco'
    13 | 'Colisión obstáculo'
     3 | 'Atropello'

=== ubicacion (2 levels) ===
  1733 | 'SUPERFICIE'
   329 | 'TÚNEL'

=== circulacion (6 levels) ==

## 2. Explicit mapping dicts + apply

A value that already appears as a single spelling is left as in the source. Maps are only for duplicate spellings of the same category, or for junk codes. Keys are accent-stripped lowercase (`norm_key`). The stored label is the most common existing spelling.

In [4]:
# Duplicate spellings only. Target = the most common string already in the column.
# tipo_colision, tipo_accidente, ubicacion, distrito: one spelling each — not mapped.

ESTADO_FIRME_MAP = {
    "seco y limpio": "Seco y limpio",
    "seco": "Seco y limpio",
    "mojado": "Mojado",
    "humedo": "Húmedo",
}

TIPO_VIA_MAP = {
    "tronco": "TRONCO",
    "via de servicio": "VIA DE SERVICIO",
    "ramal": "RAMAL",
    "transfer": "TRANSFER",
    "tunel": "Túnel",  # Túnel and TUNEL are tied (12 each); keep one existing spelling
}

# Labels already unique. Empty map still sends junk '0' / '_' → NaN.
CIRCULACION_MAP = {}

CARRETERA_MAP = {
    "m607": "M-607",
}

MAPS = {
    "estado_firme": ESTADO_FIRME_MAP,
    "tipo_via": TIPO_VIA_MAP,
    "circulacion": CIRCULACION_MAP,
    "carretera": CARRETERA_MAP,
}

for col, mapping in MAPS.items():
    if col in df.columns:
        df[col] = apply_map(df[col], mapping)

print("After categorical clean:")
for c in CLEAN_COLS:
    if c not in df.columns:
        continue
    print(f"\n=== {c} ===")
    print(df[c].value_counts(dropna=False).to_string())

After categorical clean:

=== estado_firme ===
estado_firme
Seco y limpio    1757
Mojado            210
NaN                52
Otros              32
Húmedo             11

=== tipo_via ===
tipo_via
TRONCO             850
VIA DE SERVICIO    701
RAMAL              436
TRANSFER            34
Túnel               24
TABLERO             17

=== tipo_colision ===
tipo_colision
COLISIÓN POR ALCANCE    1149
Otro                     393
COLISIÓN LATERAL         315
COLISIÓN MEDIANA          92
NaN                       80
COLISIÓN FRONTAL          33

=== tipo_accidente ===
tipo_accidente
Colision entre vehiculos    1498
Salida de via                361
Otros                        173
Vuelco                        14
Colisión obstáculo            13
Atropello                      3

=== ubicacion ===
ubicacion
SUPERFICIE    1733
TÚNEL          329

=== circulacion ===
circulacion
FLUIDA           725
CONGESTIONADA    646
DENSA            599
NaN               92

=== carretera ===
carretera
M30 

## 3. Numeric fixes and datetime parsing

- `-1` in count columns → NaN (sentinel for missing).
- `vehiculos_implicados == 0` left as-is (17 rows; may be real / pedestrian-adjacent — flag in report).
- `fecha` mixes DMY, YMD and a few MDY strings → robust parser.
- `hora` → `hour` (0–23); original `hora` kept.

In [5]:
COUNT_COLS = [
    "vehiculos_implicados", "motos_implicadas", "pesados_implicados",
    "heridos_leves", "heridos_graves", "victimas_mortales",
]

minus_one_log = {}
for c in COUNT_COLS:
    if c not in df.columns:
        continue
    df[c] = pd.to_numeric(df[c], errors="coerce")
    n_m1 = int((df[c] == -1).sum())
    minus_one_log[c] = n_m1
    if n_m1:
        df.loc[df[c] == -1, c] = np.nan

print("-1 → NaN counts:", minus_one_log)
print("vehiculos_implicados == 0:", int((df["vehiculos_implicados"] == 0).sum()))


def parse_fecha(val):
    """Parse mixed DMY / YMD / MDY date strings (date part only)."""
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip().split()[0]
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT


df["fecha_dt"] = df["fecha"].map(parse_fecha)
print(
    f"fecha_dt parsed: {df['fecha_dt'].notna().sum()} / {len(df)} "
    f"(failed {int(df['fecha_dt'].isna().sum())})"
)
print("fecha range:", df["fecha_dt"].min(), "→", df["fecha_dt"].max())

hora = pd.to_datetime(df["hora"].astype(str), format="%H:%M:%S", errors="coerce")
if hora.isna().mean() > 0.5:
    hora = pd.to_datetime(df["hora"].astype(str), format="%H:%M", errors="coerce")
df["hour"] = hora.dt.hour.astype("Int64")
print(f"hour parsed: {df['hour'].notna().sum()} / {len(df)}")

-1 → NaN counts: {'vehiculos_implicados': 15, 'motos_implicadas': 0, 'pesados_implicados': 0, 'heridos_leves': 15, 'heridos_graves': 0, 'victimas_mortales': 0}
vehiculos_implicados == 0: 17
fecha_dt parsed: 2062 / 2062 (failed 0)
fecha range: 2024-01-01 00:00:00 → 2024-12-31 00:00:00
hour parsed: 2062 / 2062


## 4. Label: `severe`

`severe = 1` if `heridos_graves > 0` **or** `victimas_mortales > 0`.

> **Leakage warning:** `heridos_leves`, `heridos_graves` and `victimas_mortales` define (or are near) the label.
> They stay in the saved CSV for auditing, but **must never be used as model features**.

In [6]:
df["severe"] = (
    (df["heridos_graves"].fillna(0) > 0) | (df["victimas_mortales"].fillna(0) > 0)
).astype(int)

n = len(df)
n_sev = int(df["severe"].sum())
print(f"severe: {n_sev} / {n} ({100 * n_sev / n:.2f}%)")
print("heridos_graves > 0:", int((df["heridos_graves"].fillna(0) > 0).sum()))
print("victimas_mortales > 0:", int((df["victimas_mortales"].fillna(0) > 0).sum()))
print("both:", int(((df["heridos_graves"].fillna(0) > 0) & (df["victimas_mortales"].fillna(0) > 0)).sum()))
df["severe"].value_counts()

severe: 45 / 2062 (2.18%)
heridos_graves > 0: 40
victimas_mortales > 0: 7
both: 2


severe
0    2017
1      45
Name: count, dtype: int64

## 5. Data report

Before/after unique counts, missingness after cleaning, label balance, and decision list.

In [7]:
df_before_cats = pd.DataFrame(snapshot_before)
report_ba = before_after(df_before_cats, df, list(snapshot_before))
print("=== Before / after unique & missing (cleaned categoricals) ===")
display(report_ba)

miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(2),
    "nunique": df.nunique(dropna=False),
    "dtype": df.dtypes.astype(str),
}).sort_values("pct_missing", ascending=False)
print("\n=== Missing % after cleaning (all columns) ===")
display(miss)

print(f"\nRows: {len(df)}")
print(f"Label balance: severe={n_sev} ({100 * n_sev / n:.2f}%), non-severe={n - n_sev} ({100 * (n - n_sev) / n:.2f}%)")

=== Before / after unique & missing (cleaned categoricals) ===


,column,nunique_before,nunique_after,missing_before,missing_after
0,estado_firme,13,5,50,52
1,tipo_via,12,6,0,0
2,tipo_colision,6,6,80,80
3,tipo_accidente,6,6,0,0
4,ubicacion,2,2,0,0
5,circulacion,6,4,75,92
6,carretera,14,13,1,1
7,distrito,25,25,0,0



=== Missing % after cleaning (all columns) ===


,n_missing,pct_missing,nunique,dtype
unidad_samur_id,558,27.06,51,float64
descripcion,112,5.43,1801,object
circulacion,92,4.46,4,object
tipo_colision,80,3.88,6,object
estado_firme,52,2.52,5,object
vehiculos_implicados,15,0.73,9,float64
heridos_leves,15,0.73,7,float64
carretera,1,0.05,13,object
fecha,0,0.00,443,object
hour,0,0.00,24,Int64



Rows: 2062
Label balance: severe=45 (2.18%), non-severe=2017 (97.82%)


## Cleaning decisions
 1. estado_firme: Seco/SECO/seco and SECO Y LIMPIO → 'Seco y limpio'; MOJADO/mojado → 'Mojado'; HUMEDO → 'Húmedo'; '(En blanco)' → NaN. 'Otros' already unique, left as-is.
 2. tipo_via: case variants folded into the majority spelling (TRONCO, VIA DE SERVICIO, RAMAL, TRANSFER). Túnel/TUNEL tied at 12 → 'Túnel'. TABLERO already unique, left as-is.
 3. tipo_colision: one spelling per level — left as in the source.
 4. tipo_accidente: one spelling per level — left as in the source (including 'Colisión obstáculo').
 5. ubicacion: SUPERFICIE and TÚNEL left as in the source.
 6. circulacion: FLUIDA / DENSA / CONGESTIONADA left as-is; junk '0' and '_' → NaN.
 7. carretera: M607 → M-607. Other codes, including AV.ANDALUCIA, left as-is.
 8. distrito: one spelling per name — left as in the source.
 9. Junk on columns we do touch: '(En blanco)', '0', '_', empty → NaN.
10. Numeric: -1 → NaN in count cols (vehiculos_implicados=15, heridos_leves=15).
11. vehiculos_implicados == 0 left unchanged (17 rows) — review in M2 if needed.
12. fecha: robust DMY / YMD / MDY parse → fecha_dt; hora → hour.
13. Label severe = (heridos_graves > 0) | (victimas_mortales > 0); injury cols kept but are leakage.

## 6. Save cleaned table

Writes `../data/interim/accidentes_clean.csv` (fallback `./accidentes_clean.csv`).

In [8]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("wrote", OUT_PATH.resolve())
print("shape", df.shape)
print("columns:", list(df.columns))
df.head(3)

wrote /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/accidentes_clean.csv
shape (2062, 24)
columns: ['fecha', 'hora', 'numero_incidencia', 'localizacion', 'ubicacion', 'carretera', 'eje', 'tipo_via', 'tipo_accidente', 'tipo_colision', 'vehiculos_implicados', 'motos_implicadas', 'pesados_implicados', 'estado_firme', 'circulacion', 'heridos_leves', 'heridos_graves', 'victimas_mortales', 'descripcion', 'distrito', 'unidad_samur_id', 'fecha_dt', 'hour', 'severe']


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id,fecha_dt,hour,severe
0,01/01/2024 0:00,1:30:48,20240101-0001,08NL10,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,NaN,1.0,0,0,SEÑALIZAR ACCIDENTE POR ALCANCE EN CARRIL CENT...,CENTRO,49.0,2024-01-01,1,0
1,01/01/2024 0:00,1:59:17,20240101-0002,07NC10,SUPERFICIE,M30,NC,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,"AVISO DE ALCANCE ENTRE 2 VEHÍCULOS, ANTES DE L...",USERA,NaN,2024-01-01,1,0
2,01/01/2024 0:00,5:58:11,20240101-0007,03FT45,TÚNEL,A5,FT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,ACCIDENTE LEVE ENTRE DOS VEHICULOS (ALCANCE) E...,VICALVARO,29.0,2024-01-01,5,0


## 7. Test: logistic regression

Quick check that the cleaned table can train a model. Not the M2 model.

Predicts `severe` from a few non-injury columns. `heridos_leves`, `heridos_graves`, `victimas_mortales` and `descripcion` are left out (leakage). Only ~2% of rows are severe, so accuracy is compared with a majority-class baseline and balanced accuracy is the number that matters.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CAT = ["tipo_accidente", "tipo_colision", "estado_firme", "ubicacion"]
NUM = ["hour", "vehiculos_implicados", "motos_implicadas", "pesados_implicados"]

X = df[CAT + NUM].copy()
for c in NUM:
    X[c] = pd.to_numeric(X[c], errors="coerce")
y = df["severe"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

pre = ColumnTransformer([
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore")),
    ]), CAT),
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), NUM),
])

clf = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

majority = int(y_train.mode().iloc[0])
baseline = np.full(len(y_test), majority)

print(f"train {len(y_train)} / test {len(y_test)}   severe in test: {int(y_test.sum())}")
print(f"features: {CAT + NUM}")
print(f"majority baseline accuracy: {accuracy_score(y_test, baseline):.3f}")
print(f"logreg accuracy:            {accuracy_score(y_test, pred):.3f}")
print(f"logreg balanced accuracy:   {balanced_accuracy_score(y_test, pred):.3f}")
print("confusion matrix [rows=true 0/1, cols=pred 0/1]:")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3, zero_division=0))

train 1546 / test 516   severe in test: 11
features: ['tipo_accidente', 'tipo_colision', 'estado_firme', 'ubicacion', 'hour', 'vehiculos_implicados', 'motos_implicadas', 'pesados_implicados']
majority baseline accuracy: 0.979
logreg accuracy:            0.781
logreg balanced accuracy:   0.666
confusion matrix [rows=true 0/1, cols=pred 0/1]:
[[397 108]
 [  5   6]]
              precision    recall  f1-score   support

           0      0.988     0.786     0.875       505
           1      0.053     0.545     0.096        11

    accuracy                          0.781       516
   macro avg      0.520     0.666     0.486       516
weighted avg      0.968     0.781     0.859       516

